# Dynamic Workflows — Claude Managed Agents (CLI)

The same flow as [`07-dynamic-workflows/`](../07-dynamic-workflows/), with the **`ant`** CLI. The lead agent **writes a workflow**, a program that runs many agents in phases, and the server executes it in the background as a **workflow run**. You follow the run on the session's stream through its phases and its agent threads.

```
workflow run  (the program the agent wrote)
 ├─ phase "Read"    →  Contract Reader × N   (a predefined agent you list, on Haiku)
 └─ phase "Verify"  →  inline checkers × N   (agents the workflow defines itself)
```

Unlike subagents (module 06), the coordination happens in the program, not in the lead's context. And **no API call starts a run**: you describe the work, and the lead decides.

> Prerequisite: the `ant` CLI (>= 1.40) installed and authenticated — see [`01-basics-cli/`](../01-basics-cli/).

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# `ant` is a separate process from this notebook, so a bit of setup:
#  - it doesn't read the root .env — load the API key into os.environ, which
#    every `!ant` call below inherits;
#  - Jupyter sets FORCE_COLOR=1, which makes ant wrap its JSON output in
#    ANSI color codes and break parsing — clear it so the output stays plain;
#  - `!` runs commands in a pseudo-terminal, so ant probes it with escape
#    sequences that leak into the cell output — TERM=dumb turns that off.
load_dotenv(find_dotenv())
os.environ.pop("FORCE_COLOR", None)
os.environ.pop("CLICOLOR_FORCE", None)
os.environ["NO_COLOR"] = "1"
os.environ["TERM"] = "dumb"
assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY not found — add it to the root .env or run `ant auth login`."
)
print("Credential loaded, color disabled — ant is ready")

In [ ]:
import json
import subprocess
import time


def ant(*args: str, input: str | None = None) -> str:
    """Run an ant command and return its stdout; show stderr if it fails."""
    result = subprocess.run(["ant", *args], input=input, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result.stdout


def ant_json(*args: str):
    return json.loads(ant(*args, "--format", "json"))


def ant_jsonl(*args: str) -> list:
    return [json.loads(line) for line in ant(*args, "--format", "jsonl").splitlines() if line.strip()]


def send(session_id: str, *events: dict) -> list:
    """Send events; returns the queued events (with their ids)."""
    return json.loads(ant("beta:sessions:events", "send", "--session-id", session_id, "--format", "json",
                          input=json.dumps({"events": list(events)})))["data"]


def open_stream(session_id: str):
    return subprocess.Popen(
        ["ant", "beta:sessions:events", "stream", "--session-id", session_id, "--format", "json"],
        stdout=subprocess.PIPE, text=True, bufsize=1,
    )


def events(stream):
    """Yield each event from `ant ... stream --format json` as a dict."""
    buf = []
    for line in stream.stdout:
        buf.append(line)
        if line.rstrip("\n") == "}":
            yield json.loads("".join(buf))
            buf = []


def message(text: str) -> dict:
    return {"type": "user.message", "content": [{"type": "text", "text": text}]}


def text_of(event: dict) -> str:
    return "".join(b["text"] for b in event.get("content", []) if b.get("type") == "text")


def cents(amount: dict) -> str:
    return f"${int(amount['amount']) / 100:.2f}"

## 2. Declare the Reader and the Lead

**`contract-reader.md`** is a predefined agent for the workflow. It's a narrow, repetitive job, so it runs on `claude-haiku-5-5`.

**`agent.md`** is the lead. Its `multiagent` block:
- `workflows: enabled` turns on dynamic workflows (they're on by default with `multiagent_20261001`; we write it out to be clear). `predefined_agents` lists the reader **by path**, and `ant apply` resolves the path to the reader's ID and version.
- `subagents: disabled`: the lead hands out work only through runs.

The **system prompt** tells the lead when to use a run, how to split the work, what to do when an agent fails, and the run's time limit.

In [ ]:
%%writefile contract-reader.md
---
name: Contract Reader
description: Reads one supplier contract and extracts its key terms.
model: claude-haiku-5-5
tools:
  - type: agent_toolset_20260401
    configs:
      - name: web_search
        enabled: false
      - name: web_fetch
        enabled: false
---

You read exactly one supplier contract and extract its key terms: supplier, effective date,
auto-renewal, non-renewal notice period, termination for convenience, liability cap and
carve-outs, change-of-control clause, governing law. Quote the clause number for every term.
Say 'not stated' when the contract is silent. Reply with the terms only.

In [ ]:
%%writefile agent.md
---
name: Contract Review Lead (workflows, CLI)
model: claude-opus-5-5
tools:
  - type: agent_toolset_20260401
multiagent:
  type: multiagent_20261001
  workflows:
    type: enabled
    predefined_agents:
      - ./contract-reader.md
  subagents:
    type: disabled
---

You review supplier contracts for Brightloom Inc. Contracts are mounted read-only under
/mnt/session/uploads/contracts/. When you're asked to review more than a few contracts, start a
workflow run instead of reading them yourself. In the run: first, one Contract Reader per contract
extracts its key terms. Then a second agent checks every extraction against its contract, not only
the ones that look wrong, and looks for anything the reader missed; an extraction that fails the
check is redone once. One agent failing must not fail the run: list that contract as not covered.
Give the run a 30-minute lifetime. When the run ends, write /mnt/session/outputs/summary.csv (one
row per contract) and /mnt/session/outputs/report.md (risks ranked from most to least urgent, citing
clause numbers). Review one or two contracts yourself, without a run. Keep chat replies short.

In [ ]:
%%writefile environment.yaml
# yaml-language-server: $schema=https://platform.claude.com/schemas/ant/beta/environment.json
name: dynamic-workflows-env-cli
config:
  type: cloud
  networking:
    type: unrestricted

In [ ]:
!ant apply agent.md environment.yaml --yes --lock-file claude-lock.json

lock = json.load(open("claude-lock.json"))["resources"]
env_id = lock["./environment.yaml"]["id"]
lead_id, lead_version = lock["./agent.md"]["id"], int(lock["./agent.md"]["version"])
print(json.dumps(ant_json("beta:agents", "retrieve", "--agent-id", lead_id, "--transform", "multiagent"), indent=2))

## 3. Mount the Contracts and Start a Budgeted Session

Every agent in a run uses tokens. The session budget caps the whole session, runs included, and a run that reaches it **pauses** until you raise it.

In [ ]:
from pathlib import Path

uploaded = {}
for path in sorted(Path("../sample-data/contracts").glob("*.md")):
    uploaded[path.name] = ant("files", "upload", "--file", str(path), "--transform", "id", "-r").strip()

resource_flags = []
for name, file_id in uploaded.items():
    resource_flags += ["--resource", json.dumps({"type": "file", "file_id": file_id, "mount_path": f"/contracts/{name}"})]

session_id = ant(
    "beta:sessions", "create",
    "--agent", json.dumps({"type": "agent", "id": lead_id, "version": lead_version}),
    "--environment-id", env_id,
    "--title", "Contract review with a workflow run CLI",
    "--budget", json.dumps({"type": "limit", "max_list_cost": {"amount": "500", "currency": "USD"}}),
    *resource_flags, "--transform", "id", "-r",
).strip()
print(f"Session: {session_id}  ({len(uploaded)} contracts, budget $5.00)")

## 4. Follow the Run

| Event | Meaning |
|---|---|
| `workflow_run.created` | A run started: `name`, `description`, and the declared `phases` (`id`, `name`) |
| `workflow_run.status_running` / `status_idle` | Executing / paused (for example at the budget) |
| `workflow_run.phase_started` / `phase_ended` | Entered / left a phase. These carry only `workflow_run_phase_id`, so look the name up in `phases`. |
| `session.thread_created` with `workflow_run_id` | The run started an agent thread |
| `workflow_run.error` | A run error, or a refused start. Log it, but don't treat it as the end. |
| `workflow_run.status_ended` | The run ended: `result.type` is `completed`, `stopped`, or `error` |

**The work is done** only when every run you've seen created has ended **and** a `session.status_idle` with `end_turn` arrives after that. While the run works in the background, the lead can end its turn and the session can go idle, so that first idle is not the end.

In [ ]:
def follow_runs(stream) -> str:
    """Follow the session until every run has ended and the lead has finished its turn."""
    open_runs, phase_names, threads, run_of = {}, {}, {}, {}
    t0 = time.monotonic()

    def log(line: str) -> None:
        print(f"{time.monotonic() - t0:6.0f}s  {line}")

    for event in events(stream):
        kind, run_id = event["type"], event.get("workflow_run_id")
        match kind:
            case "workflow_run.created":
                open_runs[run_id] = event["name"]
                threads.setdefault(run_id, [0, 0])
                for phase in event["phases"]:
                    phase_names[run_id, phase["id"]] = phase["name"]
                log(f"RUN CREATED  {event['name']!r}  phases: {[p['name'] for p in event['phases']]}")
            case "workflow_run.status_running":
                log("run running")
            case "workflow_run.status_idle":
                log("run paused (idle)")
            case "workflow_run.phase_started":
                log(f"PHASE STARTED  {phase_names.get((run_id, event['workflow_run_phase_id']), event['workflow_run_phase_id'])}")
            case "workflow_run.phase_ended":
                created, done = threads.get(run_id, [0, 0])
                log(f"phase ended    {phase_names.get((run_id, event['workflow_run_phase_id']), event['workflow_run_phase_id'])}"
                    f"   (threads so far: {done}/{created} finished)")
            case "workflow_run.error":
                log(f"run error: {event['error']['type']}: {event['error'].get('message')}")
            case "workflow_run.status_ended":
                name = open_runs.pop(run_id, run_id)
                result = event["result"]["type"]
                if result == "error":
                    result += f" ({event['result']['error']['type']})"
                log(f"RUN ENDED    {name!r}: {result}   ({threads.get(run_id, [0, 0])[0]} agent threads)")
            case "session.thread_created" if run_id:
                threads.setdefault(run_id, [0, 0])[0] += 1
                run_of[event["session_thread_id"]] = run_id
                log(f"  + thread {event['agent_name']}")
            case "session.thread_status_terminated" if event.get("session_thread_id") in run_of:
                threads[run_of[event["session_thread_id"]]][1] += 1
            case "agent.custom_tool_use":
                log(f"custom tool call {event['name']}: answer it here with user.custom_tool_result")
            case "agent.message":
                log(f"Lead: {text_of(event)}")
            case "session.status_idle":
                reason = event["stop_reason"]["type"]
                if reason == "requires_action":
                    continue
                if reason == "end_turn" and open_runs:
                    log("lead ended its turn while the run works in the background — keep following")
                    continue
                log(f"session idle: {reason}")
                return reason
            case "session.status_terminated":
                log("session terminated")
                return "terminated"


stream = open_stream(session_id)
send(session_id, message("Review all the contracts in /mnt/session/uploads/contracts/ and write the summary and the report."))
follow_runs(stream)
stream.terminate()

## 5. Inspect the Run's Threads

A run's threads carry its **`workflow_run_id`**. Predefined agents show up by name, with their `id` and `version`. Inline agents have `agent.type: inline`, plus the name the workflow gave them.

In [ ]:
threads = [t for t in ant_jsonl("beta:sessions:threads", "list", "--session-id", session_id) if t.get("workflow_run_id")]
for t in threads:
    cost = cents(t["usage"]["list_cost"]) if t.get("usage") and t["usage"].get("list_cost") else "-"
    print(f"{t['id']}  {t['agent']['type']:<8} {t['agent'].get('name', '-'):<28} {t['status']:<11} {cost}")

s = ant_json("beta:sessions", "retrieve", "--session-id", session_id)
print(f"\n{len(threads)} run threads — session total {cents(s['usage']['list_cost'])} of {cents(s['budget']['max_list_cost'])}")

reader = next((t for t in threads if t["agent"].get("name", "").startswith("Contract Reader")), threads[0])
print(f"\n--- inside {reader['agent'].get('name')} ---")
for event in ant_jsonl("beta:sessions:threads:events", "list", "--session-id", session_id, "--thread-id", reader["id"]):
    if event["type"] == "agent.thread_message_received":
        print(f"[prompt from the workflow]\n{text_of(event)[:500]}\n")
    elif event["type"] == "agent.message":
        print(f"[reply]\n{text_of(event)[:500]}")

## 6. Ask for the Workflow It Wrote

The workflow's code never appears on the stream. Once the run has ended and the session is idle, ask the agent to print it, and to not start another run.

In [ ]:
from IPython.display import Markdown, display

reply = []
stream = open_stream(session_id)
send(session_id, message(
    "Print the workflow that you started the run with, word for word, in one code block. Do not start another run."
))
for event in events(stream):
    if event["type"] == "agent.message":
        reply.append(text_of(event))
    elif event["type"] == "session.status_idle" and event["stop_reason"]["type"] != "requires_action":
        break
stream.terminate()
display(Markdown("\n".join(reply)))

## 7. Download the Deliverables and Clean Up

While a run is open, archiving or deleting the session can fail with `error_code: "workflow_run_open"`. To end runs early, ask the agent to stop them with a `user.message`; an interrupt ends no run.

In [ ]:
EXPECTED = {"summary.csv", "report.md"}
outputs = {}
for _ in range(20):
    outputs = {f["filename"]: f["id"] for f in
               ant_jsonl("beta:files", "list", "--scope-id", session_id, "--beta", "managed-agents-2026-04-01")
               if f["downloadable"]}
    if EXPECTED <= outputs.keys():
        break
    time.sleep(3)

Path("outputs").mkdir(exist_ok=True)
for name, file_id in outputs.items():
    ant("files", "download", "--file-id", file_id, "--output", f"outputs/{name}")
    print(f"downloaded outputs/{name}")

if Path("outputs/report.md").exists():
    display(Markdown(Path("outputs/report.md").read_text()))

In [ ]:
for file_id in uploaded.values():
    ant("files", "delete", "--file-id", file_id)

!ant beta:sessions archive --session-id {session_id} --transform archived_at -r

## Summary

```
contract-reader.md                    →  a predefined agent for workflows (Haiku)
agent.md  multiagent.workflows:       →  {type: enabled, predefined_agents: [./contract-reader.md]}
          system prompt               →  when to run, how to split, failure handling, time limit
ant beta:sessions create --budget ... →  one cap for the session and its runs
user.message                          →  the lead decides to start a run
workflow_run.* + session.thread_created(workflow_run_id)
done = no open runs AND session.status_idle with end_turn
```

### CLI vs SDK, same lesson

| | CLI (this module) | SDK ([`07-dynamic-workflows/`](../07-dynamic-workflows/)) |
|---|---|---|
| Turn workflows on | `multiagent:` in `agent.md`, `ant apply` | `multiagent={...}` in `agents.create()` |
| Predefined agents | a path, resolved by `ant apply` | `{"type": "agent", "id": ..., "version": ...}` |
| Follow the run | `ant ... stream --format json` + `follow_runs()` | `events.stream()` + `follow_runs()` |
| Run threads | `ant beta:sessions:threads list` | `client.beta.sessions.threads.list()` |

This is the end of the path through the cookbook. For what to try next, see the end of [`07-dynamic-workflows/`](../07-dynamic-workflows/dynamic_workflows.ipynb).